In [ ]:
# Databricks notebook source# MAGIC %md# MAGIC # Medallion Architecture: Gold Layer - Star Schema# MAGIC # MAGIC **Purpose:** Build business-ready dimensional model from Silver layer# MAGIC # MAGIC **Source:** Silver layer tables (cleaned, validated data)# MAGIC # MAGIC **Target:** Gold layer - Star Schema for Sales Analytics# MAGIC # MAGIC **Schema:**# MAGIC - **Dimensions:** Dim_Date, Dim_Customer, Dim_Product, Dim_Address# MAGIC - **Fact:** Fact_Sales# MAGIC # MAGIC **Pattern:** Type 1 SCD with structure for Type 2 evolution# COMMAND ----------# MAGIC %md# MAGIC ## Configuration# COMMAND ----------from pyspark.sql import SparkSessionfrom pyspark.sql.functions import *from pyspark.sql.types import *from pyspark.sql.window import Windowfrom datetime import datetime, timedeltaimport logginglogging.basicConfig(level=logging.INFO)logger = logging.getLogger(__name__)TARGET_LAKEHOUSE = "SalesAnalytics"SILVER_LAYER = "silver"GOLD_LAYER = "gold"def get_silver_table(table_name):    return f"{TARGET_LAKEHOUSE}.{SILVER_LAYER}_{table_name}"def get_gold_table(table_name):    return f"{GOLD_LAYER}_{table_name}"logger.info("Gold layer configuration loaded")# COMMAND ----------# MAGIC %md# MAGIC ## 1. Build gold_Dim_Date# COMMAND ----------def generate_date_dimension(start_date='2000-01-01', end_date='2030-12-31'):    """Generate comprehensive date dimension"""    logger.info("Generating gold_Dim_Date...")        from datetime import datetime, timedelta    start = datetime.strptime(start_date, '%Y-%m-%d')    end = datetime.strptime(end_date, '%Y-%m-%d')        date_list = []    current = start    while current <= end:        date_list.append((current.strftime('%Y-%m-%d'),))        current += timedelta(days=1)        date_df = spark.createDataFrame(date_list, ["DateString"])        dim_date = date_df.select(        to_date(col("DateString")).alias("Date")    ).withColumn(        "DateKey", date_format(col("Date"), "yyyyMMdd").cast("int")    ).withColumn(        "Year", year(col("Date"))    ).withColumn(        "Quarter", quarter(col("Date"))    ).withColumn(        "QuarterName", concat(lit("Q"), col("Quarter"))    ).withColumn(        "Month", month(col("Date"))    ).withColumn(        "MonthName", date_format(col("Date"), "MMMM")    ).withColumn(        "MonthNameShort", date_format(col("Date"), "MMM")    ).withColumn(        "DayOfMonth", dayofmonth(col("Date"))    ).withColumn(        "DayOfWeek", ((dayofweek(col("Date")) + 5) % 7) + 1  # ISO 8601: Mon=1    ).withColumn(        "DayOfWeekName", date_format(col("Date"), "EEEE")    ).withColumn(        "DayOfWeekNameShort", date_format(col("Date"), "EEE")    ).withColumn(        "DayOfYear", dayofyear(col("Date"))    ).withColumn(        "WeekOfYear", weekofyear(col("Date"))    ).withColumn(        "IsWeekend", col("DayOfWeek").isin([6, 7])    ).withColumn(        "IsHoliday", lit(False)    ).withColumn(        "FiscalYear", col("Year")    ).withColumn(        "FiscalQuarter", col("Quarter")    ).withColumn(        "FiscalMonth", col("Month")    )        dim_date = dim_date.select(        "DateKey", "Date", "Year", "Quarter", "QuarterName",        "Month", "MonthName", "MonthNameShort",        "DayOfMonth", "DayOfWeek", "DayOfWeekName", "DayOfWeekNameShort",        "DayOfYear", "WeekOfYear", "IsWeekend", "IsHoliday",        "FiscalYear", "FiscalQuarter", "FiscalMonth"    )        row_count = dim_date.count()    logger.info(f"  Generated {row_count:,} date rows")        return dim_datedim_date = generate_date_dimension()dim_date.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Date')}")logger.info("✓ gold_Dim_Date created")# COMMAND ----------# MAGIC %md# MAGIC ## 2. Build gold_Dim_Customer# COMMAND ----------def build_gold_customer():    """Build gold customer dimension from silver"""    logger.info("Building gold_Dim_Customer...")        silver_customer = spark.read.table(get_silver_table("Customer"))        # Transform to dimension    dim_customer = silver_customer.select(        col("CustomerID"),        col("Title"),        col("Suffix"),        col("CompanyName"),        col("SalesPerson"),        col("EmailAddress"),        col("ModifiedDate")    )        # Add surrogate key    window_spec = Window.orderBy("CustomerID")    dim_customer = dim_customer.withColumn(        "CustomerKey", row_number().over(window_spec)    )        # Add SCD Type 2 structure    current_date_val = current_date()    dim_customer = dim_customer \        .withColumn("IsCurrent", lit(True)) \        .withColumn("EffectiveDate", current_date_val) \        .withColumn("EndDate", lit(None).cast("date")) \        .withColumn("FirstName", lit(None).cast("string")) \        .withColumn("LastName", lit(None).cast("string"))        dim_customer = dim_customer.select(        "CustomerKey", "CustomerID", "Title", "FirstName", "LastName", "Suffix",        "CompanyName", "SalesPerson", "EmailAddress", "ModifiedDate",        "IsCurrent", "EffectiveDate", "EndDate"    )        row_count = dim_customer.count()    logger.info(f"  Built {row_count:,} customer dimension rows")        return dim_customerdim_customer = build_gold_customer()dim_customer.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Customer')}")logger.info("✓ gold_Dim_Customer created")# COMMAND ----------# MAGIC %md# MAGIC ## 3. Build gold_Dim_Product# COMMAND ----------def build_gold_product():    """Build gold product dimension with denormalized hierarchy"""    logger.info("Building gold_Dim_Product...")        silver_product = spark.read.table(get_silver_table("Product"))    silver_category = spark.read.table(get_silver_table("ProductCategory"))    silver_model = spark.read.table(get_silver_table("ProductModel"))        # Self-join category for parent    parent_category = silver_category.select(        col("ProductCategoryID").alias("ParentCategoryID"),        col("Name").alias("ParentProductCategoryName")    )        category_with_parent = silver_category \        .join(parent_category,               silver_category.ParentProductCategoryID == parent_category.ParentCategoryID,              "left") \        .select(            silver_category.ProductCategoryID,            col("Name").alias("ProductCategoryName"),            silver_category.ParentProductCategoryID,            "ParentProductCategoryName"        )        # Join Product → Category → Model    dim_product = silver_product \        .join(category_with_parent, "ProductCategoryID", "left") \        .join(silver_model.select(col("ProductModelID"), col("Name").alias("ProductModelName")),               "ProductModelID", "left")        # Add calculated fields    dim_product = dim_product \        .withColumn("IsDiscontinued", col("DiscontinuedDate").isNotNull()) \        .withColumn("ProductName", lit(None).cast("string"))  # Will be enriched if description tables available        # Add surrogate key    window_spec = Window.orderBy("ProductID")    dim_product = dim_product.withColumn(        "ProductKey", row_number().over(window_spec)    )        # Add SCD Type 2 structure    current_date_val = current_date()    dim_product = dim_product \        .withColumn("IsCurrent", lit(True)) \        .withColumn("EffectiveDate", current_date_val) \        .withColumn("EndDate", lit(None).cast("date"))        dim_product = dim_product.select(        "ProductKey", "ProductID", "ProductNumber", "ProductName",        "Color", "Size", "Weight", "StandardCost", "ListPrice",        "ProductCategoryID", "ProductCategoryName",        "ParentProductCategoryID", "ParentProductCategoryName",        "ProductModelID", "ProductModelName",        "SellStartDate", "SellEndDate", "DiscontinuedDate", "IsDiscontinued",        "ModifiedDate", "IsCurrent", "EffectiveDate", "EndDate"    )        row_count = dim_product.count()    logger.info(f"  Built {row_count:,} product dimension rows")        return dim_productdim_product = build_gold_product()dim_product.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Product')}")logger.info("✓ gold_Dim_Product created")# COMMAND ----------# MAGIC %md# MAGIC ## 4. Build gold_Dim_Address# COMMAND ----------def build_gold_address():    """Build gold address dimension"""    logger.info("Building gold_Dim_Address...")        silver_address = spark.read.table(get_silver_table("Address"))        dim_address = silver_address.select(        col("AddressID"),        col("AddressLine1"),        col("AddressLine2"),        col("City"),        col("StateProvince"),        col("CountryRegion"),        col("PostalCode"),        col("ModifiedDate")    )        # Add surrogate key    window_spec = Window.orderBy("AddressID")    dim_address = dim_address.withColumn(        "AddressKey", row_number().over(window_spec)    )        # Add SCD Type 2 structure    current_date_val = current_date()    dim_address = dim_address \        .withColumn("IsCurrent", lit(True)) \        .withColumn("EffectiveDate", current_date_val) \        .withColumn("EndDate", lit(None).cast("date"))        # Create "Unknown" address    unknown_address = spark.createDataFrame([        (-1, -1, "Unknown", None, "Unknown", None, None, "00000",          datetime.now(), True, datetime.now().date(), None)    ], ["AddressKey", "AddressID", "AddressLine1", "AddressLine2", "City",        "StateProvince", "CountryRegion", "PostalCode", "ModifiedDate",        "IsCurrent", "EffectiveDate", "EndDate"])        dim_address = dim_address.select(        "AddressKey", "AddressID", "AddressLine1", "AddressLine2", "City",        "StateProvince", "CountryRegion", "PostalCode", "ModifiedDate",        "IsCurrent", "EffectiveDate", "EndDate"    ).union(unknown_address)        row_count = dim_address.count()    logger.info(f"  Built {row_count:,} address dimension rows")        return dim_addressdim_address = build_gold_address()dim_address.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Address')}")logger.info("✓ gold_Dim_Address created")# COMMAND ----------# MAGIC %md# MAGIC ## 5. Build gold_Fact_Sales# COMMAND ----------def build_gold_fact_sales():    """Build gold sales fact table"""    logger.info("Building gold_Fact_Sales...")        # Read silver fact tables    silver_detail = spark.read.table(get_silver_table("SalesOrderDetail"))    silver_header = spark.read.table(get_silver_table("SalesOrderHeader"))    silver_product = spark.read.table(get_silver_table("Product"))        # Read gold dimensions for lookups    dim_customer_lookup = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Customer')}")    dim_product_lookup = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Product')}")    dim_date_lookup = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Date')}")    dim_address_lookup = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Address')}")        # Join detail with header    fact_sales = silver_detail.join(silver_header, "SalesOrderID")        # Join with product for standard cost    fact_sales = fact_sales.join(        silver_product.select("ProductID", col("StandardCost").alias("ProductStandardCost")),        "ProductID"    )        # Calculate measures (if not already in silver)    fact_sales = fact_sales \        .withColumn("LineTotal", col("LineTotal")) \        .withColumn("LineProfit", col("LineTotal") - (col("OrderQty") * col("ProductStandardCost"))) \        .withColumn("OrderTotal", col("TotalDue"))        # Lookup CustomerKey    fact_sales = fact_sales.join(        dim_customer_lookup.select("CustomerID", "CustomerKey"),        "CustomerID", "inner"    )        # Lookup ProductKey    fact_sales = fact_sales.join(        dim_product_lookup.select("ProductID", "ProductKey"),        "ProductID", "inner"    )        # Lookup OrderDateKey    fact_sales = fact_sales.join(        dim_date_lookup.select(col("Date").cast("date").alias("OrderDate"),                                col("DateKey").alias("OrderDateKey")),        to_date(fact_sales.OrderDate) == col("OrderDate"), "inner"    ).drop("OrderDate")        # Lookup DueDateKey    fact_sales = fact_sales.join(        dim_date_lookup.select(col("Date").cast("date").alias("DueDate"),                               col("DateKey").alias("DueDateKey")),        to_date(fact_sales.DueDate) == col("DueDate"), "inner"    ).drop("DueDate")        # Lookup ShipDateKey (nullable)    fact_sales = fact_sales.join(        dim_date_lookup.select(col("Date").cast("date").alias("ShipDate"),                               col("DateKey").alias("ShipDateKey")),        to_date(fact_sales.ShipDate) == col("ShipDate"), "left"    ).drop("ShipDate")        # Lookup ShipToAddressKey (nullable)    fact_sales = fact_sales.join(        dim_address_lookup.select(col("AddressID").alias("ShipToAddressID"),                                  col("AddressKey").alias("ShipToAddressKey")),        "ShipToAddressID", "left"    )        # Lookup BillToAddressKey (nullable)    fact_sales = fact_sales.join(        dim_address_lookup.select(col("AddressID").alias("BillToAddressID"),                                  col("AddressKey").alias("BillToAddressKey")),        "BillToAddressID", "left"    )        # Add surrogate key    window_spec = Window.orderBy("SalesOrderID", "SalesOrderDetailID")    fact_sales = fact_sales.withColumn(        "SalesKey", row_number().over(window_spec)    )        # Select final columns    fact_sales = fact_sales.select(        "SalesKey",        "SalesOrderID",        "SalesOrderDetailID",        "CustomerKey",        "ProductKey",        "OrderDateKey",        "DueDateKey",        "ShipDateKey",        "ShipToAddressKey",        "BillToAddressKey",        "OrderQty",        "UnitPrice",        "UnitPriceDiscount",        "LineTotal",        "ProductStandardCost",        "LineProfit",        col("SubTotal").alias("OrderSubTotal"),        col("TaxAmt").alias("OrderTaxAmt"),        col("Freight").alias("OrderFreight"),        "OrderTotal",        col("Status").alias("OrderStatus"),        "ShipMethod",        "RevisionNumber"    )        row_count = fact_sales.count()    logger.info(f"  Built {row_count:,} fact rows")        return fact_salesfact_sales = build_gold_fact_sales()fact_sales.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.{get_gold_table('Fact_Sales')}")logger.info("✓ gold_Fact_Sales created")# COMMAND ----------# MAGIC %md# MAGIC ## 6. Gold Layer Validation# COMMAND ----------logger.info("\n" + "=" * 70)logger.info("GOLD LAYER SUMMARY")logger.info("=" * 70)# Read all gold tablesgold_dim_customer = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Customer')}")gold_dim_product = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Product')}")gold_dim_date = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Date')}")gold_dim_address = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Dim_Address')}")gold_fact_sales = spark.read.table(f"{TARGET_LAKEHOUSE}.{get_gold_table('Fact_Sales')}")logger.info(f"gold_Dim_Date: {gold_dim_date.count():,} rows")logger.info(f"gold_Dim_Customer: {gold_dim_customer.count():,} rows")logger.info(f"gold_Dim_Product: {gold_dim_product.count():,} rows")logger.info(f"gold_Dim_Address: {gold_dim_address.count():,} rows")logger.info(f"gold_Fact_Sales: {gold_fact_sales.count():,} rows")# Business metricstotal_revenue = gold_fact_sales.agg(sum("LineTotal")).collect()[0][0]total_profit = gold_fact_sales.agg(sum("LineProfit")).collect()[0][0]total_orders = gold_fact_sales.select("SalesOrderID").distinct().count()logger.info("\nBUSINESS METRICS")logger.info("-" * 70)logger.info(f"Total Orders: {total_orders:,}")  logger.info(f"Total Revenue: ${total_revenue:,.2f}")logger.info(f"Total Profit: ${total_profit:,.2f}")logger.info(f"Profit Margin: {(total_profit/total_revenue*100):.2f}%")logger.info("\n✓ Gold star schema complete!")# COMMAND ----------# MAGIC %md# MAGIC ## 7. Sample Analytics Queries# COMMAND ----------# MAGIC %sql# MAGIC -- Revenue by year# MAGIC SELECT # MAGIC     d.Year,# MAGIC     COUNT(DISTINCT f.SalesOrderID) as Orders,# MAGIC     SUM(f.LineTotal) as Revenue,# MAGIC     SUM(f.LineProfit) as Profit# MAGIC FROM SalesAnalytics.gold_Fact_Sales f# MAGIC JOIN SalesAnalytics.gold_Dim_Date d ON f.OrderDateKey = d.DateKey# MAGIC GROUP BY d.Year# MAGIC ORDER BY d.Year;